In [34]:
from logging import getLogger
from itertools import product
from utils.dataset import RecDataset
from utils.dataloader import TrainDataLoader, EvalDataLoader
from utils.logger import init_logger
from utils.configurator import Config
from utils.utils import init_seed, get_model, get_trainer, dict2str
import platform
import os

In [35]:
model = 'PGL'
dataset = 'baby'
config_dict = {
    'gpu_id': 0,
}
save_model = True
mg = False
config = Config(model, dataset, config_dict, mg)
init_logger(config)
logger = getLogger()

hyper_ret = []
val_metric = config['valid_metric'].lower()
best_test_value = 0.0
idx = best_test_idx = 0
hyper_ls = []
if "seed" not in config['hyper_parameters']:
    config['hyper_parameters'] = ['seed'] + config['hyper_parameters']
for i in config['hyper_parameters']:
    hyper_ls.append(config[i] or [None])
# combinations
combinators = list(product(*hyper_ls))
total_loops = len(combinators)

for j, k in zip(config['hyper_parameters'], combinators[0]):
    config[j] = k
init_seed(config['seed'])




In [36]:
MODEL_FILE_PATH = r'D:\Show_me_everything\Amazon-Recommend-System\saved_model\Dec-18-2025-11-42-46\best_model.pth'

In [37]:
def load_system():
    print(f"--> Đang load dataset '{config['dataset']}' từ '{config['data_path']}'...")
    # Bước này quan trọng: Load dữ liệu để tạo Graph
    
    dataset = RecDataset(config)
    train_dataset, valid_dataset, test_dataset = dataset.split()
    logger.info('\n====Training====\n' + str(train_dataset))
    logger.info('\n====Validation====\n' + str(valid_dataset))
    logger.info('\n====Testing====\n' + str(test_dataset))
    train_data = TrainDataLoader(config, train_dataset, batch_size=config['train_batch_size'], shuffle=True)
    (valid_data, test_data) = (
        EvalDataLoader(config, valid_dataset, additional_dataset=train_dataset, batch_size=config['eval_batch_size']),
        EvalDataLoader(config, test_dataset, additional_dataset=train_dataset, batch_size=config['eval_batch_size']))
    
    print("--> Đang khởi tạo model PGL...")
    model = get_model(config['model'])(config, train_data).to(config['device'])
    
    print(f"--> Đang load trọng số từ: {MODEL_FILE_PATH}")
    if os.path.exists(MODEL_FILE_PATH):
        # Load state dict
        state_dict = torch.load(MODEL_FILE_PATH, map_location=config['device'])
        model.load_state_dict(state_dict)
        
        # Chuyển sang device & chế độ eval
        model = model.to(config['device'])
        model.eval()
        print("--> Load thành công!")
    else:
        raise FileNotFoundError(f"Không tìm thấy file model: {MODEL_FILE_PATH}")
        
    return model, dataset

In [38]:
load_system()

18 Dec 12:03    INFO 
====Training====
baby
The number of users: 19445
Average actions of users: 6.096734379017742
The number of items: 7047
Average actions of items: 16.82290336313325
The number of inters: 118551
The sparsity of the dataset: 99.9134846831415%
18 Dec 12:03    INFO 
====Validation====
baby
The number of users: 19445
Average actions of users: 1.0572897917202366
The number of items: 5483
Average actions of items: 3.749589640707642
The number of inters: 20559
The sparsity of the dataset: 99.98071694707788%
18 Dec 12:03    INFO 
====Testing====
baby
The number of users: 19445
Average actions of users: 1.1150424273592183
The number of items: 5549
Average actions of items: 3.907370697422959
The number of inters: 21682
The sparsity of the dataset: 99.97990552482683%


--> Đang load dataset 'baby' từ '../data/'...
--> Đang khởi tạo model PGL...
--> Đang load trọng số từ: D:\Show_me_everything\Amazon-Recommend-System\saved_model\Dec-18-2025-11-42-46\best_model.pth
--> Load thành công!


(PGL(
   (user_text): Embedding(19445, 64)
   (user_image): Embedding(19445, 64)
   (image_embedding): Embedding(7050, 4096)
   (image_trs): Linear(in_features=4096, out_features=64, bias=True)
   (text_embedding): Embedding(7050, 384)
   (text_trs): Linear(in_features=384, out_features=64, bias=True)
   (dropoutf): Dropout(p=0.2, inplace=False)
 ),
 baby
 The number of users: 19445
 Average actions of users: 8.269066598097197
 The number of items: 7050
 Average actions of items: 22.807375886524824
 The number of inters: 160792
 The sparsity of the dataset: 99.8827082752043%)